In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 81.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=7497ec8f0daa728658960a9c5168be31d762494c65ee49cee24339f7a0b634d2
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [3]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

# HELPER: Quantum Random Number Generator

# All randomness is produced by measuring qubits prepared in |+> = (1/sqrt(2))(|0> + |1>).  Measuring this state yields
# 0 or 1 with equal probability -- a true quantum coin flip.

simulator = BasicSimulator()

def quantum_random_bits(n):
    """Generate n random bits by measuring qubits each prepared in |+>."""
    # BasicSimulator supports max 24 bits
    MAX_BATCH = 24
    bits = []
    while len(bits) < n:
        batch = min(MAX_BATCH, n - len(bits))
        qc = QuantumCircuit(batch, batch)
        for i in range(batch):
            qc.h(i)   # |0> --> |+> = (|0> + |1>) / sqrt(2)
        qc.measure(range(batch), range(batch))
        compiled = transpile(qc, simulator)
        job = simulator.run(compiled, shots=1)
        counts = job.result().get_counts()
        bitstring = list(counts.keys())[0].replace(' ', '')
        # Qiskit stores results with qubit 0 as the rightmost character
        bits.extend([int(b) for b in reversed(bitstring)])
    return bits[:n]



# ALICE: Choose random bits and bases, then encode each qubit

# Basis 0 = Z (standard, +):  |0> and |1>
# Basis 1 = X (diagonal,    x):  |+> and |->

NUM_QUBITS  = 50   # larger sample to make Eve's statistical footprint visible

alice_bits  = quantum_random_bits(NUM_QUBITS)
alice_bases = quantum_random_bits(NUM_QUBITS)

print('=== ALICE ===')
print(f'Bits:   {alice_bits}')
print(f'Bases:  {alice_bases}   (0 = Z/+,  1 = X/x)')

def alice_encode(bit, basis):
    """Return a 1-qubit circuit encoding `bit` in `basis`.

    Z basis: bit=0 -> |0>,  bit=1 -> |1>
    X basis: bit=0 -> |+>,  bit=1 -> |->
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

encoded_qubits = [alice_encode(alice_bits[i], alice_bases[i])
                  for i in range(NUM_QUBITS)]

=== ALICE ===
Bits:   [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0]
Bases:  [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1]   (0 = Z/+,  1 = X/x)


In [4]:

# attacker: Intercept-and-resend attack

# attacker sits on the quantum channel between Alice and Bob.
# For each qubit she:
#   1. Picks a random measurement basis (quantum coin flip).
#   2. Measures Alice's qubit -- this COLLAPSES the quantum state.
#   3. Re-prepares a new qubit in her measured basis/result and forwards it.
#
# When attacker's basis matches Alice's, the forwarded qubit is correct.
# When they differ, the forwarded qubit is in the wrong basis, so Bob
# gets the wrong answer ~50% of those cases --> ~25% overall error rate
# on matching (Alice,Bob) basis pairs, which reveals the attack.

attacker_bases   = quantum_random_bits(NUM_QUBITS)
attacker_results = []

def attacker_intercept(qc, attacker_basis):
    """attacker measures `qc` in `attacker_basis` and re-prepares the qubit.

    Returns (new_circuit, measured_bit).
    """
    # Measure in attacker's chosen basis
    qc_attacker = qc.copy()
    if attacker_basis == 1:
        qc_attacker.h(0)
    qc_attacker.measure(0, 0)
    compiled = transpile(qc_attacker, simulator)
    job = simulator.run(compiled, shots=1)
    counts = job.result().get_counts()
    attacker_bit = int(list(counts.keys())[0])

    # Reprepare and forward a qubit encoding attacker's measurement outcome
    qc_new = QuantumCircuit(1, 1)
    if attacker_bit == 1:
        qc_new.x(0)
    if attacker_basis == 1:
        qc_new.h(0)

    return qc_new, attacker_bit

intercepted_qubits = []
for i in range(NUM_QUBITS):
    fwd_qc, attacker_bit = attacker_intercept(encoded_qubits[i], attacker_bases[i])
    intercepted_qubits.append(fwd_qc)
    attacker_results.append(attacker_bit)

print('=== attacker ===')
print(f'Bases:    {attacker_bases}   (0 = Z/+,  1 = X/x)')
print(f'Results:  {attacker_results}')
print('(attacker has intercepted and re-transmitted all qubits)')

=== attacker ===
Bases:    [1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0]   (0 = Z/+,  1 = X/x)
Results:  [0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1]
(attacker has intercepted and re-transmitted all qubits)


In [5]:

# BOB: Measure the (intercepted) qubits

# Bob receives attacker's re-prepared qubits, not Alice's originals.

bob_bases = quantum_random_bits(NUM_QUBITS)

def bob_measure(qc, basis):
    """Bob measures qubit from circuit `qc` in `basis`."""
    qc_m = qc.copy()
    if basis == 1:
        qc_m.h(0)
    qc_m.measure(0, 0)
    compiled = transpile(qc_m, simulator)
    job = simulator.run(compiled, shots=1)
    counts = job.result().get_counts()
    return int(list(counts.keys())[0])

bob_results = [bob_measure(intercepted_qubits[i], bob_bases[i])
               for i in range(NUM_QUBITS)]

print('=== BOB ===')
print(f'Bases:    {bob_bases}   (0 = Z/+,  1 = X/x)')
print(f'Results:  {bob_results}')

=== BOB ===
Bases:    [1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0]   (0 = Z/+,  1 = X/x)
Results:  [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1]


In [6]:

# SIFTING: Alice and Bob publicly compare bases


matching     = [i for i in range(NUM_QUBITS) if alice_bases[i] == bob_bases[i]]
alice_sifted = [alice_bits[i]  for i in matching]
bob_sifted   = [bob_results[i] for i in matching]

print('=== SIFTING ===')
print(f'Matching positions: {matching}')
print(f'Alice sifted key:   {alice_sifted}')
print(f'Bob   sifted key:   {bob_sifted}')
print(f'Sifted key length:  {len(alice_sifted)} bits  ',
      f'(expected ~{NUM_QUBITS // 2})')


# VERIFICATION: Compare a sample to detect Eve

# When Eve uses the wrong basis (~50% of cases), the qubit she
# forwards is disturbed.  When Bob then measures in Alice's basis,
# he gets the wrong answer ~50% of those times.
# Net error rate on matching positions: ~25% -- well above threshold.

SAMPLE_SIZE = max(1, len(alice_sifted) // 4)

sample_alice = alice_sifted[:SAMPLE_SIZE]
sample_bob   = bob_sifted[:SAMPLE_SIZE]
errors       = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate   = errors / SAMPLE_SIZE

print()
print('=== VERIFICATION ===')
print(f'Check bits ({SAMPLE_SIZE}): Alice={sample_alice}  Bob={sample_bob}')
print(f'Errors: {errors}/{SAMPLE_SIZE}  -->  error rate = {error_rate:.1%}')

THRESHOLD = 0.25

if error_rate > THRESHOLD:
    print(f'ATTACK DETECTED -- error rate {error_rate:.1%} exceeds '
          f'threshold {THRESHOLD:.1%}. Aborting key exchange.')
else:
    # Eve was lucky this run; warn but still report
    print(f'Attack not detected this run (error rate {error_rate:.1%} '
          f'<= threshold {THRESHOLD:.1%}) -- Eve was lucky.')
    final_key = alice_sifted[SAMPLE_SIZE:]
    print(f'Tentative key ({len(final_key)} bits): {final_key}')
    print('NOTE: Eve may still have partial knowledge of this key.')

# ANALYSIS: Show how often attacker guessed the right basis

attacker_correct_basis = sum(
    1 for i in matching if attacker_bases[i] == alice_bases[i]
)
print()
print('=== ATTACKER ANALYSIS (hidden from Alice and Bob) ===')
print(f'Attacker guessed Alice\'s basis correctly: '
      f'{attacker_correct_basis}/{len(matching)} sifted positions '
      f'({attacker_correct_basis/len(matching):.1%})')
print(f'Expected errors introduced by attacker:   '
      f'~{len(matching)//4} bits (~25%)')

=== SIFTING ===
Matching positions: [4, 5, 7, 8, 9, 11, 12, 14, 17, 21, 23, 25, 28, 29, 31, 33, 34, 36, 37, 38, 39, 40, 42, 43, 47, 48]
Alice sifted key:   [1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0]
Bob   sifted key:   [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1]
Sifted key length:  26 bits   (expected ~25)

=== VERIFICATION ===
Check bits (6): Alice=[1, 1, 1, 0, 1, 1]  Bob=[1, 1, 1, 1, 1, 1]
Errors: 1/6  -->  error rate = 16.7%
Attack not detected this run (error rate 16.7% <= threshold 25.0%) -- Eve was lucky.
Tentative key (20 bits): [1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0]
NOTE: Eve may still have partial knowledge of this key.

=== ATTACKER ANALYSIS (hidden from Alice and Bob) ===
Attacker guessed Alice's basis correctly: 15/26 sifted positions (57.7%)
Expected errors introduced by attacker:   ~6 bits (~25%)
